# 04 · Decision trees (readable rules)

A single tree is a set of if-then rules — easy to explain, but unstable and its probabilities come in steps.
The benchmarking doc cites an ED study where a tree found 87% of unsuccessful payments. Trees need no scaling.

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import loguniform, uniform, randint
from salud_pipeline.phase2.propensity import modeling as M
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200); pd.set_option("display.precision", 3)

DESIGN = "fixed"   # training rows: "fixed" (one per patient on the scoring date; notebook 00 §6) or "call"
WORLD = "salud"    # synthetic world: "salud" (tuned to Salud's data, salud-synth-calibrate) or "original"
dev, holdout, real = M.load_data(design=DESIGN, world=WORLD)
SPLITS = M.cv_splits(dev)          # the same 25 folds for every run in every notebook -> paired comparisons
NB = "04"
def run(rid, pipe, fs, desc, **kw):
    return M.run_experiment(rid, pipe, dev, fs, notebook=NB, description=desc, splits=SPLITS, **kw)
print(f"{DESIGN} design: {len(dev):,} training rows, pay rate {dev.paid_30d.mean():.2f}; 25 folds")

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.calibration import CalibratedClassifierCV
DT = lambda **k: DecisionTreeClassifier(random_state=M.SEED, **k)
GRID = {"decisiontreeclassifier__max_depth": randint(2, 9), "decisiontreeclassifier__min_samples_leaf": randint(10, 120),
        "decisiontreeclassifier__ccp_alpha": loguniform(1e-5, 2e-2)}
run("DT-01", M.pipeline(M.preprocess("none"), DT(max_depth=3)), "FS_ALL", "Depth 3 (readable)")
run("DT-02", M.pipeline(M.preprocess("none"), DT(max_depth=5, min_samples_leaf=50)), "FS_ALL", "Depth 5, min leaf 50")
run("DT-03", M.tuned(M.pipeline(M.preprocess("none"), DT()), GRID, n_iter=25), "FS_ALL", "Tuned: depth, min leaf, cost-complexity pruning")
run("DT-04", M.tuned(M.pipeline(M.preprocess("none"), DT()), GRID, n_iter=25), "FS_TRANSFER", "Tuned, FS_TRANSFER")
cal = M.pipeline(M.preprocess("none"), CalibratedClassifierCV(DT(max_depth=5, min_samples_leaf=50), method="isotonic", cv=3))
run("DT-05", cal, "FS_TRANSFER", "Depth 5 + isotonic calibration, FS_TRANSFER");

In [ ]:
board = M.leaderboard(notebook=NB)
cols = ["run_id", "description", "feature_set", "auc", "auc_sd", "brier", "cal_slope", "top20", "n_features", "fit_s"]
board[cols]

In [ ]:
r = M.read_results(); r = r[r.notebook == NB]
plt.figure(figsize=(10, 0.35 * r.run_id.nunique() + 1.2))
sns.boxplot(data=r, y="run_id", x="auc", order=board.run_id, color="#8DA0CB")
plt.axvline(M.leaderboard().query("run_id == 'REF-LR'").auc.iloc[0], ls="--", c="#C44E52", label="REF-LR (notebook 01)")
plt.legend(); plt.title("AUC across the same 25 folds"); plt.ylabel(""); plt.show()

In [ ]:
best = board.run_id.iloc[0]
pd.DataFrame([M.corrected_ttest(best, rid) for rid in board.run_id[1:]]).round(4)

## The depth-3 tree (all training rows)

In [ ]:
X, y = M.xy(dev, M.FS_TRANSFER)
t = M.pipeline(M.preprocess("none"), DT(max_depth=3, min_samples_leaf=30)).fit(X, y)
plt.figure(figsize=(18, 7)); plot_tree(t[-1], feature_names=list(t[:-1].get_feature_names_out()), class_names=["no pay", "pay"],
                                       filled=True, proportion=True, impurity=False, fontsize=9); plt.show()

## Findings (Salud-tuned world)
- A depth-5 tree (DT-02, 0.877) is level with logistic regression. Tuning (DT-03) adds nothing (p = 0.65).
  A readable depth-3 tree (DT-01) reaches 0.860.
- Raw trees are over-confident (calibration slope about 0.78). Isotonic calibration (DT-05) fixes it (slope 1.00, Brier 0.115) without losing AUC.